In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import matplotlib.pyplot as plt

DATA = '../data/case-study/processed'

# load and merge
session_dfs = []
for s in [1, 2, 3]:
    hr = pd.read_csv(f'{DATA}/hr_{s:02d}.csv')
    ibi = pd.read_csv(f'{DATA}/ibi_{s:02d}.csv')
    sed = pd.read_csv(f'{DATA}/sed_{s:02d}.csv')

    hr_clean = hr[hr['confidence'] == 1.0]
    ibi_clean = ibi[ibi['ibi'] > 0]

    hr_agg = hr_clean.groupby('datetime')['heart_rate'].mean().reset_index()
    ibi_agg = ibi_clean.groupby('datetime')['ibi'].mean().reset_index()

    merged = pd.merge(hr_agg, ibi_agg, on='datetime')
    merged = pd.merge(merged, sed, on='datetime')
    session_dfs.append(merged)

all_data_combined = pd.concat(session_dfs, ignore_index=True)

print("Columns available in DataFrame:", all_data_combined.columns)

# define features
features = [
    'heart_rate', 'ibi', 'headPos.x', 'headPos.y', 'headPos.z',
    'gazeDir.x', 'gazeDir.y', 'gazeDir.z', 'pupil'
]

existing_features = [feature for feature in features if feature in all_data_combined.columns]

missing_features = set(features) - set(existing_features)
if missing_features:
    print(f"Missing columns in the DataFrame: {missing_features}")

all_data_combined = all_data_combined.dropna(subset=existing_features)

# standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(all_data_combined[existing_features])

# pca for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

# find optimal k
silhouette_scores = []
K = range(2, 11)
fitted_models = {}
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    fitted_models[k] = km
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

plt.figure(figsize=(10, 6))
plt.plot(K, silhouette_scores, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs. Number of Clusters')
plt.show()
plt.close()

# auto-select optimal k
optimal_k = list(K)[np.argmax(silhouette_scores)]
print(f"Optimal k selected: {optimal_k} (silhouette score: {max(silhouette_scores):.4f})")

kmeans = fitted_models[optimal_k]
clusters = kmeans.labels_

# pca cluster plot
plt.figure(figsize=(10, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='tab10', marker='o')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title(f'K-Means Clusters (k={optimal_k}) in PCA-Reduced Space')
plt.colorbar(label='Cluster')
plt.show()
plt.close()

# cluster centers
cluster_centers = scaler.inverse_transform(kmeans.cluster_centers_)
cluster_sizes = pd.Series(clusters).value_counts()

cluster_df = pd.DataFrame(cluster_centers, columns=existing_features).round(3)
print("Cluster Centers:")
print(cluster_df.to_string())
print("\nCluster Sizes:\n", cluster_sizes)

# quality metrics
dbi = davies_bouldin_score(X_scaled, clusters)
chi = calinski_harabasz_score(X_scaled, clusters)

print(f"\nDavies-Bouldin Index: {dbi:.4f}")
print(f"Calinski-Harabasz Index: {chi:.4f}")